# NOTEBOOK FEATURE ENGINEERING

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy.stats import chi2_contingency
from itertools import combinations
from scipy.stats import f_oneway

import os
import re

from IPython.display import display, Markdown

import missingno as msno
import sys

from rapidfuzz import process, fuzz

from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

from itertools import product

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r"C:\Users\JAIME\Documents\Github\TFM-Sesgos-en-el-sistema-judicial-de-EEUU-\00_Data\00_Processed\df_eda.csv")

In [3]:
df.columns

Index(['sex', 'race', 'age', 'age_cat', 'decile_score', 'v_decile_score',
       'is_recid', 'is_violent_recid', 'score_text', 'screening_date',
       'two_year_recid', 'priors_count', 'c_charge_degree', 'start', 'end',
       'event', 'juv_fel_count', 'juv_misd_count', 'juv_other_count',
       'person_id', 'agency_text', 'maritalstatus', 'language', 'rawscore',
       'juv_priors_count'],
      dtype='object')

In [4]:
lista_variables_modelo = [
    'person_id',
    'decile_score',
    'rawscore',
    'v_decile_score',
    'is_recid',
    'is_violent_recid',
    'two_year_recid',
    'sex',
    'race',
    'age',
    'priors_count',
    'juv_priors_count',
    'c_charge_degree',
    'maritalstatus'
]

In [5]:
df = df[lista_variables_modelo]

In [6]:
df.columns

Index(['person_id', 'decile_score', 'rawscore', 'v_decile_score', 'is_recid',
       'is_violent_recid', 'two_year_recid', 'sex', 'race', 'age',
       'priors_count', 'juv_priors_count', 'c_charge_degree', 'maritalstatus'],
      dtype='object')

In [7]:
df['race'] = df['race'].replace({'asian': 'other', 'native american': 'other'})

In [8]:
df.race.value_counts()

race
african-american    3096
caucasian           2100
hispanic             561
other                380
Name: count, dtype: int64

In [9]:
df['sex'] = df['sex'].replace({'male': 0, 'female': 1})

In [10]:
df.sex.value_counts()

sex
0    4941
1    1196
Name: count, dtype: int64

In [11]:
df['c_charge_degree'] = df['c_charge_degree'].replace({'felony': 0, 'misdemeanor': 1})

In [12]:
df.c_charge_degree.value_counts()

c_charge_degree
0    3884
1    2253
Name: count, dtype: int64

In [13]:
df['maritalstatus'] = df['maritalstatus'].replace({'married': 'significant other', 'divorced': 'separated', 'widowed': 'other', 'unknown': 'other'})

In [14]:
df.maritalstatus.value_counts()

maritalstatus
single               4736
significant other     939
separated             408
other                  54
Name: count, dtype: int64

In [15]:
def one_hot_encoding(df, column, drop_val):
    encoder = OneHotEncoder(
    drop=[drop_val], 
    sparse_output=False
    )

    encoded = encoder.fit_transform(df[[column]])

    df_encoded = pd.DataFrame(
    encoded,
    columns = encoder.get_feature_names_out([column])
    )

    return df_encoded

In [16]:
df_marital_status = pd.concat([df, one_hot_encoding(df, 'maritalstatus', 'single')], axis = 1)

In [17]:
df_no_caucasian = pd.concat([df_marital_status, one_hot_encoding(df, 'race', 'caucasian')], axis = 1)

In [18]:
df_no_african = pd.concat([df_marital_status, one_hot_encoding(df, 'race', 'african-american')], axis = 1)

In [19]:
df.columns

Index(['person_id', 'decile_score', 'rawscore', 'v_decile_score', 'is_recid',
       'is_violent_recid', 'two_year_recid', 'sex', 'race', 'age',
       'priors_count', 'juv_priors_count', 'c_charge_degree', 'maritalstatus'],
      dtype='object')

In [20]:
lista_columnas_dropear = ['person_id', 'decile_score', 'rawscore', 'v_decile_score', 'is_recid', 'is_violent_recid', 'maritalstatus', 'race']       #revisar una vez esté todo claro

In [21]:
df_no_caucasian = df_no_caucasian.drop(columns = lista_columnas_dropear)

In [24]:
df_no_caucasian.columns

Index(['two_year_recid', 'sex', 'age', 'priors_count', 'juv_priors_count',
       'c_charge_degree', 'maritalstatus_other', 'maritalstatus_separated',
       'maritalstatus_significant other', 'race_african-american',
       'race_hispanic', 'race_other'],
      dtype='object')

In [22]:
df_no_african = df_no_african.drop(columns = lista_columnas_dropear)

In [23]:
df.to_csv(r'C:\Users\JAIME\Documents\Github\TFM-Sesgos-en-el-sistema-judicial-de-EEUU-\00_Data\00_Processed\df_estudio.csv', index=False)

df_no_caucasian.to_csv(r'C:\Users\JAIME\Documents\Github\TFM-Sesgos-en-el-sistema-judicial-de-EEUU-\00_Data\00_Processed\df_no_caucasian.csv', index=False)

df_no_african.to_csv(r'C:\Users\JAIME\Documents\Github\TFM-Sesgos-en-el-sistema-judicial-de-EEUU-\00_Data\00_Processed\df_no_african.csv', index=False)